
# 01 — Data Preparation

## Deep Learning in Asset Pricing — Replication Project

**Reference:** Chen, L., Pelger, M., & Zhu, J. (2023), *Deep Learning in Asset Pricing*, Management Science, 70(2), 714–750.

### Purpose of this notebook

This notebook builds the reproducible data pipeline used by the rest of the replication project. It is intentionally separated from model training so that every later model—linear benchmark, feedforward network, LSTM, and GAN—uses exactly the same cleaned and time-split inputs.

### Main tasks

1. Load the stock-return / firm-characteristic panel.
2. Load the macroeconomic time series.
3. Identify and validate the date, stock identifier, return, and characteristic fields.
4. Reproduce the paper-style complete-case stock sample.
5. Cross-sectionally rank-normalize firm characteristics.
6. Apply the fixed train / validation / test split.
7. Run data-quality checks.
8. Save processed datasets for later notebooks.

> **Important:** This notebook does not train an asset-pricing model. Its only job is to make the input data correct, transparent, and reproducible.



## 1. Replication targets from the paper

The paper's empirical U.S. equity application uses:

- Monthly U.S. stock returns from **January 1967 through December 2016**.
- **46 time-varying firm characteristics**.
- **178 macroeconomic time series**.
- Stocks with complete firm-characteristic information in a given month for the core empirical sample.
- Cross-sectional rank / quantile normalization of firm characteristics.
- A fixed chronological split:
  - **Training:** 1967–1986
  - **Validation:** 1987–1991
  - **Test:** 1992–2016

The macroeconomic information described in the paper combines FRED-MD variables, cross-sectional characteristic medians, and additional equity-premium predictors. If the supplied `Macro.csv` is already the authors' processed macro file, this notebook will preserve it rather than re-transforming the series.

### Expected local files

For this replication, place the author-provided or project-provided files in:

```text
data/raw/
```

The default filenames used below are:

```text
RetChar.csv
Macro.csv
```

If your files have different names, change only the configuration cell below.


## 2. Imports and Reproducibility

This section loads the Python libraries required for the data-preparation stage of the replication.

The main packages used here are:

- `pathlib` for working with project directories and file paths,
- `numpy` for numerical operations,
- `pandas` for loading, cleaning, and manipulating tabular data,
- `random` and NumPy's random-number generator for reproducibility,
- `warnings` for controlling unnecessary warning messages.

A fixed random seed is also defined:

\[
\text{SEED} = 42.
\]

Although this notebook does not yet train a neural network, setting the seed at the beginning of the project is good reproducibility practice. Later notebooks will use random initialization, minibatch sampling, and model training procedures that depend on random-number generation.

The display settings are adjusted so that large financial datasets and model inputs can be inspected more conveniently inside Jupyter.

Finally, the installed NumPy and pandas versions are printed so that the computational environment can be documented and reproduced.

In [9]:

# 2. Imports and reproducibility

from pathlib import Path
import random
import warnings

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda x: f"{x:,.6f}")

print("NumPy :", np.__version__)
print("pandas:", pd.__version__)
print("Seed  :", SEED)


NumPy : 2.5.3
pandas: 3.0.5
Seed  : 42


## 3. Locate the Repository Root

A reproducible research project should not depend on hard-coded computer-specific paths such as

`/Users/reza/Desktop/...`

because those paths will change across users, computers, and operating systems.

Instead, this section automatically locates the root directory of the GitHub repository.

The function searches upward from the current working directory until it finds either:

- the `.git` directory, or
- the `requirements.txt` file.

Once the repository root is identified, the notebook defines the main project directories:

- `data/raw/` for original input datasets,
- `data/processed/` for cleaned and transformed datasets,
- `notebooks/` for Jupyter notebooks.

This allows the same notebook to work whether Jupyter is launched from the repository root or from the `notebooks` directory.

The approach also makes the replication easier for collaborators because no user-specific file paths need to be changed.

In [10]:

# 3. Locate the repository root robustly

def find_repo_root(start: Path | None = None) -> Path:
    """Search upward for the project root using requirements.txt or .git."""
    p = (start or Path.cwd()).resolve()
    for candidate in [p, *p.parents]:
        if (candidate / "requirements.txt").exists() or (candidate / ".git").exists():
            return candidate
    raise FileNotFoundError(
        "Could not locate the repository root. "
        "Open Jupyter from inside the cloned project repository."
    )

ROOT = find_repo_root()
RAW_DIR = ROOT / "data" / "raw"
PROCESSED_DIR = ROOT / "data" / "processed"
NOTEBOOK_DIR = ROOT / "notebooks"

RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
NOTEBOOK_DIR.mkdir(parents=True, exist_ok=True)

print("Repository root :", ROOT)
print("Raw data         :", RAW_DIR)
print("Processed data   :", PROCESSED_DIR)


Repository root : /Users/reza/Desktop/mfe-term 3/deepl learning1/project /github/Deep-Learning-in-Asset-Pricing_-Replication-project
Raw data         : /Users/reza/Desktop/mfe-term 3/deepl learning1/project /github/Deep-Learning-in-Asset-Pricing_-Replication-project/data/raw
Processed data   : /Users/reza/Desktop/mfe-term 3/deepl learning1/project /github/Deep-Learning-in-Asset-Pricing_-Replication-project/data/processed


## 4. Replication Configuration

This section defines the main configuration choices used throughout the data-preparation pipeline.

The expected raw input files are:

- `RetChar.csv` for stock returns and firm characteristics,
- `Macro.csv` for macroeconomic variables.

Keeping filenames and sample definitions in one configuration section makes the notebook easier to modify and audit.

The replication uses the chronological sample split described in the paper:

\[
\text{Training: 1967--1986}
\]

\[
\text{Validation: 1987--1991}
\]

\[
\text{Test: 1992--2016}
\]

This time-based split is important in asset-pricing applications because financial models should be evaluated using observations that occur strictly after the data used for model estimation.

Using a random train-test split would allow information from future periods to enter the training sample and would therefore introduce look-ahead bias.

The variable `STRICT_PAPER_REPLICATION` is included to distinguish the core replication from possible later robustness or extension exercises.

In [11]:

# 4. Configuration

RETCHAR_FILE = "RetChar.csv"
MACRO_FILE = "Macro.csv"

# Core paper-style sample treatment:
STRICT_PAPER_REPLICATION = True

# Fixed chronological split
TRAIN_START = pd.Timestamp("1967-01-01")
TRAIN_END   = pd.Timestamp("1986-12-31")

VALID_START = pd.Timestamp("1987-01-01")
VALID_END   = pd.Timestamp("1991-12-31")

TEST_START  = pd.Timestamp("1992-01-01")
TEST_END    = pd.Timestamp("2016-12-31")

print("Configured stock file:", RETCHAR_FILE)
print("Configured macro file:", MACRO_FILE)
print("Strict paper replication:", STRICT_PAPER_REPLICATION)


Configured stock file: RetChar.csv
Configured macro file: Macro.csv
Strict paper replication: True


## 5. Inspect the Raw-Data Directory

Before loading the datasets, this section checks which files are currently available in the project's `data/raw/` directory.

This serves several purposes.

First, it confirms that the required input files have been placed in the correct location.

Second, it reports the approximate size of each file. This is useful because the stock-characteristic dataset is large and may require substantially more memory and loading time than the macroeconomic dataset.

Third, performing this check before model construction helps make the data pipeline transparent. If an expected file is missing, the notebook reports the problem before any preprocessing is attempted.

The raw files are kept separate from processed datasets so that the original data remain unchanged throughout the replication.

In [12]:

# 5. Inventory the raw-data directory

raw_files = sorted([p for p in RAW_DIR.iterdir() if p.is_file() and p.name != ".gitkeep"])

if raw_files:
    display(pd.DataFrame({
        "file": [p.name for p in raw_files],
        "size_MB": [round(p.stat().st_size / 1024**2, 3) for p in raw_files],
    }))
else:
    print(
        "No raw data files found yet.\n"
        f"Place {RETCHAR_FILE} and {MACRO_FILE} in:\n{RAW_DIR}"
    )


,file,size_MB
0,Macro.csv,1.732000
1,RetChar.csv,"1,099.296000"


## 6. Load the Raw Datasets

This section defines a reusable function for loading tabular datasets.

The loader supports several common file formats:

- CSV,
- Excel,
- Parquet.

For this replication, the two principal input files are CSV files:

`RetChar.csv`

and

`Macro.csv`.

The function first checks whether each file exists. If a file cannot be found, it reports the missing filename rather than immediately terminating the entire notebook.

When the file is successfully loaded, the function reports the number of rows and columns.

For the supplied replication data, we obtain:

- `RetChar.csv`: 1,218,555 observations and 48 columns,
- `Macro.csv`: 600 observations and 179 columns.

These dimensions provide an important initial validation of the datasets.

The macro dataset contains 600 monthly observations, corresponding to 50 years of monthly data from 1967 through 2016.

The 179 macro columns are consistent with one date column plus 178 macroeconomic predictors.

The stock-characteristic file contains 48 columns, which we will verify in the next section correspond to one date variable, one return variable, and 46 firm characteristics.

In [13]:
# 6. Generic tabular loader

def load_table(path: Path) -> pd.DataFrame | None:
    if not path.exists():
        print(f"[MISSING] {path.name}")
        return None

    suffix = path.suffix.lower()

    if suffix == ".csv":
        df = pd.read_csv(path)
    elif suffix in {".xlsx", ".xls"}:
        df = pd.read_excel(path)
    elif suffix in {".parquet", ".pq"}:
        df = pd.read_parquet(path)
    else:
        raise ValueError(f"Unsupported file type: {suffix}")

    print(
        f"[LOADED] {path.name}: "
        f"{df.shape[0]:,} rows × {df.shape[1]:,} columns"
    )

    return df


retchar_raw = load_table(RAW_DIR / RETCHAR_FILE)
macro_raw = load_table(RAW_DIR / MACRO_FILE)

[LOADED] RetChar.csv: 1,218,555 rows × 48 columns
[LOADED] Macro.csv: 600 rows × 179 columns



## 7. Standardize the stock / characteristic panel

The original input is expected to contain one row per stock-month with:

- a date,
- a stock identifier such as `permno`,
- a monthly return such as `ret`,
- and 46 firm characteristics.

The helper functions below detect common naming variants. We still print the detected fields so that the mapping is explicit and can be corrected if necessary.


## 7. Standardize and Inspect the Stock-Characteristic Data

### 7A. Helper Functions for Column Names and Dates

Before analyzing the stock-characteristic dataset, we define several utility functions that standardize its structure.

The first function cleans column names by:

- removing leading and trailing spaces,
- converting names to lowercase,
- replacing spaces and hyphens with underscores.

Standardized column names reduce the risk of errors caused by small formatting differences across datasets.

The second function searches for important variables, such as the date or return column, using a set of possible names.

This makes the notebook more flexible when working with files whose naming conventions differ slightly.

The third function converts the date variable into a consistent monthly `datetime` format.

For example, a date represented as

\[
196701
\]

is converted to

\[
\text{1967-01-01}.
\]

All dates are standardized to the beginning of each month. This makes it easier to merge stock data with macroeconomic variables and to construct chronological training, validation, and test samples.

In [14]:
# 7A. Column-name and date helpers

def clean_column_names(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.columns = [
        str(c).strip().lower().replace(" ", "_").replace("-", "_")
        for c in out.columns
    ]
    return out


def detect_column(columns, candidates, label):
    cols = list(columns)
    for c in candidates:
        if c in cols:
            return c
    raise KeyError(
        f"Could not detect the {label} column. "
        f"Tried {candidates}. Available columns begin with: {cols[:20]}"
    )


def parse_monthly_date(s: pd.Series) -> pd.Series:
    """
    Parse monthly dates robustly.

    Supports:
    - YYYYMM, e.g. 196701
    - M/D/YY, e.g. 1/1/67
    - standard datetime strings

    For two-digit years in this replication:
    67-99 -> 1967-1999
    00-66 -> 2000-2066
    """

    text = s.astype(str).str.strip()

    out = pd.Series(
        pd.NaT,
        index=s.index,
        dtype="datetime64[ns]"
    )

    # -----------------------------------
    # Case 1: YYYYMM
    # Example: 196701
    # -----------------------------------
    yyyymm_mask = text.str.fullmatch(r"\d{6}")

    if yyyymm_mask.any():
        out.loc[yyyymm_mask] = pd.to_datetime(
            text.loc[yyyymm_mask],
            format="%Y%m",
            errors="coerce"
        )

    # -----------------------------------
    # Case 2: M/D/YY
    # Example: 1/1/67
    # -----------------------------------
    short_date_mask = text.str.fullmatch(
        r"\d{1,2}/\d{1,2}/\d{2}"
    )

    if short_date_mask.any():

        parts = text.loc[short_date_mask].str.extract(
            r"(\d{1,2})/(\d{1,2})/(\d{2})"
        )

        month = parts[0].astype(int)
        year2 = parts[2].astype(int)

        # Explicit century rule for the 1967-2016 sample
        full_year = np.where(
            year2 >= 67,
            1900 + year2,
            2000 + year2
        )

        corrected_dates = pd.to_datetime(
            pd.DataFrame({
                "year": full_year,
                "month": month,
                "day": 1
            }, index=parts.index)
        )

        out.loc[short_date_mask] = corrected_dates

    # -----------------------------------
    # Case 3: Other standard date formats
    # -----------------------------------
    remaining = ~(yyyymm_mask | short_date_mask)

    if remaining.any():
        out.loc[remaining] = pd.to_datetime(
            text.loc[remaining],
            errors="coerce"
        )

    # Standardize all dates to month start
    return out.dt.to_period("M").dt.to_timestamp()

### 7B. Identify the Structure of `RetChar.csv`

After standardizing column names and dates, we inspect the actual structure of the supplied stock-characteristic dataset.

The file contains:

- one monthly date variable,
- one stock-return variable,
- 46 firm characteristics.

Unlike a raw CRSP stock panel, this supplied processed file does not contain a stock identifier such as `PERMNO`.

Therefore, the replication does not attempt to construct stock identities from this file. Instead, each row is treated as one stock-month observation from the processed dataset supplied for the replication.

The dataset contains:

\[
1,218,555
\]

stock-month observations covering

\[
600
\]

months from January 1967 through December 2016.

The 48-column structure can therefore be written as

\[$
=
1\text{ date}
+
1\text{ return}
+
46\text{ firm characteristics}.$
\]

Verifying this structure is an important replication checkpoint because the original empirical model uses 46 firm-level characteristics as inputs to the stochastic discount factor network.

At this stage, we also inspect several observations from the dataset to verify that the variables have been loaded correctly before applying any additional preprocessing.

In [15]:
# 7B. Normalize and inspect the stock panel

if retchar_raw is not None:
    retchar = clean_column_names(retchar_raw)

    DATE_COL = detect_column(
        retchar.columns,
        ["date", "yyyymm", "month", "caldt", "time"],
        "date",
    )

    RET_COL = detect_column(
        retchar.columns,
        ["ret", "return", "ret_excess", "excess_ret", "retx"],
        "return",
    )

    # This supplied RetChar.csv does not contain a stock identifier.
    ID_COL = None

    retchar[DATE_COL] = parse_monthly_date(retchar[DATE_COL])
    retchar[RET_COL] = pd.to_numeric(retchar[RET_COL], errors="coerce")

    # In this dataset, every numeric column other than date and return
    # should be a firm characteristic.
    characteristic_cols = [
        c for c in retchar.columns
        if c not in [DATE_COL, RET_COL]
        and pd.api.types.is_numeric_dtype(retchar[c])
    ]

    print("Detected date column       :", DATE_COL)
    print("Detected return column     :", RET_COL)
    print("Stock identifier column    : Not provided in this dataset")
    print("Firm characteristics       :", len(characteristic_cols))
    print("Date range                 :", retchar[DATE_COL].min(), "to", retchar[DATE_COL].max())
    print("Total observations         :", f"{len(retchar):,}")
    print("Unique months              :", retchar[DATE_COL].nunique())

    display(retchar.head())

Detected date column       : date
Detected return column     : ret
Stock identifier column    : Not provided in this dataset
Firm characteristics       : 46
Date range                 : 1967-01-01 00:00:00 to 2016-12-01 00:00:00
Total observations         : 1,218,555
Unique months              : 600


,date,ret,a2me,ac,at,ato,beme,beta,c,cf,cf2p,cto,d2a,d2p,dpi2a,e2p,fc2y,idiovol,investment,lev,lme,lt_rev,lturnover,mktbeta,ni,noa,oa,ol,op,pcm,pm,prof,q,r2_1,r12_2,r12_7,r36_13,rel2high,resid_var,rna,roa,roe,s2p,sga2s,spread,st_rev,suv,variance
0,1967-01-01,0.136223,0.108392,0.243590,0.152681,-0.182984,0.113054,0.185315,0.113054,-0.271562,0.404429,-0.252914,0.036131,0.150350,0.001166,0.343823,-0.381119,-0.155012,0.073427,-0.010490,0.010490,0.432401,0.059441,-0.136364,-0.390443,0.134033,0.259907,-0.245921,0.143357,-0.148019,0.311189,-0.283217,-0.217949,-0.008159,-0.043124,0.250583,0.106061,-0.122378,-0.224942,0.045455,0.059441,0.080420,-0.015152,-0.383450,-0.164336,-0.241259,-0.325175,-0.182984
1,1967-01-01,0.317129,0.455711,-0.360140,-0.472028,0.257576,0.453380,0.318182,0.122378,0.071096,0.430070,0.180653,-0.194639,-0.500000,-0.488345,0.488345,-0.201632,0.495338,-0.467366,0.173660,-0.488345,-0.439394,0.332168,-0.145688,-0.320513,-0.294872,-0.334499,0.355478,-0.357809,-0.409091,-0.392774,-0.068765,0.015152,0.082751,0.364802,0.495338,-0.329837,-0.490676,0.483683,0.038462,-0.346154,-0.245921,0.439394,-0.175991,0.439394,-0.399767,-0.064103,0.483683
2,1967-01-01,0.101064,0.050117,0.050117,0.206294,-0.276224,0.096737,0.068765,-0.294872,-0.325175,0.374126,-0.325175,0.099068,0.148019,-0.115385,0.019814,-0.029138,-0.369464,-0.182984,0.003497,0.103730,-0.057110,0.138695,-0.085082,0.175991,0.269231,0.047786,-0.280886,-0.110723,0.124709,0.203963,-0.194639,-0.187646,-0.334499,-0.294872,-0.320513,-0.047786,0.206294,-0.129371,-0.145688,-0.136364,-0.134033,-0.108392,0.001166,0.115385,0.500000,0.273893,0.026807
3,1967-01-01,0.287367,-0.031469,0.019814,0.416084,-0.252914,0.005828,-0.297203,-0.493007,-0.313520,0.315851,-0.320513,0.416084,-0.208625,0.213287,-0.054779,-0.054779,-0.413753,0.024476,-0.078089,0.339161,-0.297203,0.150350,-0.336830,-0.166667,0.236597,0.001166,-0.313520,0.040793,0.092075,0.150350,-0.236597,-0.108392,-0.399767,-0.341492,-0.378788,-0.423077,-0.271562,-0.460373,-0.136364,-0.059441,-0.099068,-0.173660,-0.178322,0.024476,-0.001166,0.378788,-0.357809
4,1967-01-01,0.143427,0.371795,0.497669,0.280886,-0.175991,0.362471,0.325175,-0.346154,-0.080420,0.148019,-0.141026,-0.129371,-0.168998,-0.297203,0.120047,-0.213287,0.110723,-0.085082,-0.120047,-0.003497,0.180653,0.388112,0.378788,-0.318182,0.243590,0.483683,-0.038462,-0.320513,-0.332168,-0.304196,-0.238928,-0.385781,-0.383450,-0.427739,-0.162005,0.374126,-0.416084,0.071096,-0.278555,-0.392774,-0.383450,0.299534,-0.189977,0.292541,0.152681,-0.304196,0.248252


## 8. Stock-Panel Data Quality Checks

Before applying any additional preprocessing, we first verify the structure and quality of the supplied stock-characteristic dataset.

The `RetChar.csv` file contains:

- one monthly date variable,
- one stock-return variable,
- 46 firm-level characteristics.

The dataset spans January 1967 through December 2016, which corresponds to 600 monthly observations in the time dimension.

Because this file appears to be a processed replication dataset, it is important to inspect the characteristics before transforming them again. In particular, we check:

1. the number of observations,
2. the number of months,
3. missing values in returns,
4. missing values in the 46 firm characteristics,
5. the overall range of the characteristic values.

The range check is especially important because the paper transforms firm characteristics using cross-sectional rank normalization. If the supplied characteristics are already bounded approximately between -0.5 and 0.5, this is strong evidence that the transformation has already been applied.

In that case, applying rank normalization a second time would unnecessarily modify the supplied replication data.

In [16]:
# 8. Stock-panel data quality report

if retchar_raw is not None:

    stock_quality = pd.Series({
        "rows": len(retchar),
        "columns": retchar.shape[1],
        "unique_months": retchar[DATE_COL].nunique(),
        "missing_dates": retchar[DATE_COL].isna().sum(),
        "missing_returns": retchar[RET_COL].isna().sum(),
        "firm_characteristics": len(characteristic_cols),
    }, name="value")

    display(stock_quality.to_frame())

    # Missingness by characteristic
    char_missing = (
        retchar[characteristic_cols]
        .isna()
        .mean()
        .sort_values(ascending=False)
        .rename("missing_fraction")
        .to_frame()
    )

    display(char_missing.head(20))

    # Overall characteristic range
    char_min = retchar[characteristic_cols].min().min()
    char_max = retchar[characteristic_cols].max().max()

    print(f"Overall characteristic minimum: {char_min:.6f}")
    print(f"Overall characteristic maximum: {char_max:.6f}")

    if len(characteristic_cols) == 46:
        print("✓ Exactly 46 firm characteristics detected.")
    else:
        print(
            f"WARNING: detected {len(characteristic_cols)} characteristics; "
            "the paper uses 46."
        )

,value
rows,1218555
columns,48
unique_months,600
missing_dates,0
missing_returns,0
firm_characteristics,46


,missing_fraction
a2me,0.000000
r36_13,0.000000
ol,0.000000
op,0.000000
pcm,0.000000
pm,0.000000
prof,0.000000
q,0.000000
r2_1,0.000000
r12_2,0.000000


Overall characteristic minimum: -0.500000
Overall characteristic maximum: 0.500000
✓ Exactly 46 firm characteristics detected.


### Check Whether the Firm Characteristics Are Already Rank-Normalized

The original paper converts firm characteristics into cross-sectional ranks.

For each month, stocks are ranked according to each characteristic, and the resulting values are centered so that they lie approximately in the interval

\[
[-0.5,\,0.5].
\]

Conceptually, the transformation is

\[
\widetilde{x}_{i,t}
=
\text{RankPercentile}_{t}(x_{i,t}) - 0.5.
\]

This transformation has several advantages:

- characteristics measured in very different units become comparable,
- extreme raw values have less influence on the neural network,
- the model focuses on the relative cross-sectional position of each stock,
- the input scale is stable across time.

However, the supplied `RetChar.csv` appears to already contain values in approximately this range.

Therefore, before performing any new normalization, we explicitly test whether the characteristics are already bounded between approximately -0.5 and 0.5.

If they are, we preserve the supplied values rather than rank-normalizing them again.

In [17]:
# Check whether characteristics are already rank-normalized

overall_min = retchar[characteristic_cols].min().min()
overall_max = retchar[characteristic_cols].max().max()

already_rank_normalized = (
    overall_min >= -0.500001
    and overall_max <= 0.500001
)

print(f"Characteristic minimum: {overall_min:.6f}")
print(f"Characteristic maximum: {overall_max:.6f}")
print("Appears already rank-normalized:", already_rank_normalized)

Characteristic minimum: -0.500000
Characteristic maximum: 0.500000
Appears already rank-normalized: True


## 9. Preserve the Supplied Rank-Normalized Firm Characteristics

The diagnostic checks above show that the 46 firm characteristics are already scaled approximately within the interval

\[
[-0.5, 0.5].
\]

This is consistent with the cross-sectional rank normalization used in the paper.

Because the supplied `RetChar.csv` already appears to contain the processed characteristic values, we do not apply any additional rank transformation.

Applying the ranking procedure a second time would modify the supplied replication inputs and could create unnecessary differences relative to the intended data construction.

Therefore, the firm-characteristic values are preserved exactly as provided in `RetChar.csv`, subject only to subsequent validation, missing-value checks, and sample-period filtering.

In [18]:
# 9. Preserve the supplied rank-normalized characteristics

if retchar_raw is not None:
    stock_processed = retchar.copy()

    # Final verification
    char_min = stock_processed[characteristic_cols].min().min()
    char_max = stock_processed[characteristic_cols].max().max()

    print(f"Characteristic minimum: {char_min:.6f}")
    print(f"Characteristic maximum: {char_max:.6f}")
    print("No additional rank normalization applied.")

Characteristic minimum: -0.500000
Characteristic maximum: 0.500000
No additional rank normalization applied.


In [19]:
# Check remaining missing values before the sample split

missing_returns = stock_processed[RET_COL].isna().sum()
missing_characteristics = stock_processed[characteristic_cols].isna().sum().sum()

print("Missing returns:", missing_returns)
print("Missing characteristic cells:", missing_characteristics)

Missing returns: 0
Missing characteristic cells: 0


### 9A. Preserve the Processed Firm-Characteristic Panel

The supplied `RetChar.csv` already contains the 46 firm characteristics in processed, rank-normalized form.

Therefore, this step does not transform the characteristics again. Instead, it creates the working stock-level dataset that will be used in the remaining stages of the replication.

The purpose of this step is to:

- preserve the supplied characteristic values exactly as provided,
- retain the monthly return variable,
- retain the standardized monthly date variable,
- create a clean working copy of the stock-characteristic panel,
- verify that the characteristic range remains consistent with the expected approximately \([-0.5, 0.5]\) scaling.

This is important because any unnecessary re-transformation could alter the replication inputs and create differences relative to the processed dataset intended for the model.

The resulting dataset, `stock_processed`, will be used for the chronological train, validation, and test split in later sections.

In [20]:
# 9A. Preserve the supplied processed firm-characteristic panel

if retchar_raw is not None:

    # Create the working copy without re-transforming the characteristics
    stock_processed = retchar.copy()

    # Basic dataset information
    print(f"Total observations: {len(stock_processed):,}")
    print(f"Unique months: {stock_processed[DATE_COL].nunique():,}")

    # Verify the supplied characteristic range
    char_min = stock_processed[characteristic_cols].min().min()
    char_max = stock_processed[characteristic_cols].max().max()

    print(f"Characteristic minimum: {char_min:.6f}")
    print(f"Characteristic maximum: {char_max:.6f}")

    print("No additional rank normalization applied.")

Total observations: 1,218,555
Unique months: 600
Characteristic minimum: -0.500000
Characteristic maximum: 0.500000
No additional rank normalization applied.


### 9B. Check for Missing Values Before the Sample Split

Before dividing the data into training, validation, and test periods, we check whether any missing values remain in the variables that will be used by the model.

The key variables are:

- the monthly stock return,
- the 46 firm characteristics.

This check is important because missing observations can cause problems during model estimation and may also change the effective sample if they are handled inconsistently across periods.

At this stage, we report:

- the number of missing return observations,
- the total number of missing characteristic values across all 46 firm characteristics.

If no missing values remain, the processed panel can move directly to the chronological sample split.

If missing values are present, they should be handled explicitly and documented before model training rather than being silently imputed or dropped later.

In [21]:
# 9B. Check for missing values before the sample split

if retchar_raw is not None:

    missing_dates = stock_processed[DATE_COL].isna().sum()
    missing_returns = stock_processed[RET_COL].isna().sum()

    missing_characteristics = (
        stock_processed[characteristic_cols]
        .isna()
        .sum()
        .sum()
    )

    print(f"Missing dates: {missing_dates:,}")
    print(f"Missing returns: {missing_returns:,}")
    print(f"Missing characteristic cells: {missing_characteristics:,}")

Missing dates: 0
Missing returns: 0
Missing characteristic cells: 0


In [22]:
stock_processed = retchar.copy()

### Interpretation

The missing-value check confirms that the processed stock-characteristic panel contains no missing observations in the variables required for the replication.

Specifically:

- there are no missing monthly dates,
- there are no missing stock returns,
- there are no missing values across the 46 firm characteristics.

Therefore, no observations need to be removed and no imputation procedure is required at this stage.

This is consistent with the interpretation that `RetChar.csv` is already a cleaned and processed replication dataset.

The full stock-characteristic panel can therefore be preserved and passed directly to the chronological train, validation, and test split.


## 10. Prepare the macroeconomic panel

The paper uses a high-dimensional macroeconomic information set. The original transformations are series-specific because many macro series must be made stationary.

For this first replication notebook:

- If `Macro.csv` is the processed file distributed for the paper/project, we preserve its values.
- We standardize its date field and verify time coverage.
- We **do not** apply a generic transformation to all macro variables because that would not faithfully reproduce the paper's series-specific transformations.

If we later discover that the macro file contains raw rather than transformed series, the transformation map should be implemented explicitly in a separate preprocessing function.


### 10A. Standardize and Inspect the Macroeconomic Dataset

The `Macro.csv` file contains the aggregate economic information used in the replication.

The dataset has:

- 600 monthly observations,
- one date column,
- 178 macroeconomic predictors.

This section first standardizes the macro dataset by cleaning the column names and converting the date variable into the same monthly datetime format used for the stock-characteristic panel.

The purpose of this step is to verify that the macroeconomic data:

- cover the intended sample period from 1967 through 2016,
- contain the expected number of macro predictors,
- have one observation per month,
- can be aligned correctly with the stock-return data.

Unlike the firm characteristics, the macro variables are not automatically transformed again at this stage. The supplied `Macro.csv` is treated as a processed replication input unless later diagnostics indicate otherwise.

This preserves the original macroeconomic information used for the replication and avoids introducing additional transformations that may differ from the paper's intended preprocessing.

In [23]:

# 10A. Clean and inspect macro data

if macro_raw is not None:
    macro = clean_column_names(macro_raw)

    MACRO_DATE_COL = detect_column(
        macro.columns,
        ["date", "yyyymm", "month", "caldt", "time"],
        "macro date",
    )
    macro[MACRO_DATE_COL] = parse_monthly_date(macro[MACRO_DATE_COL])

    macro_feature_cols = [
        c for c in macro.columns
        if c != MACRO_DATE_COL and pd.api.types.is_numeric_dtype(macro[c])
    ]

    print("Macro date column :", MACRO_DATE_COL)
    print("Macro features    :", len(macro_feature_cols))
    print("Date range        :", macro[MACRO_DATE_COL].min(), "to", macro[MACRO_DATE_COL].max())
    print("Duplicate months  :", macro.duplicated([MACRO_DATE_COL]).sum())

    if len(macro_feature_cols) != 178:
        print(
            f"NOTE: detected {len(macro_feature_cols)} numeric macro features. "
            "The full paper information set contains 178 macroeconomic series. "
            "Confirm whether this file is already reduced/processed or includes nonnumeric fields."
        )

    display(macro.head())
else:
    print("Skipped: macro panel has not been loaded.")


Macro date column : date
Macro features    : 178
Date range        : 1967-01-01 00:00:00 to 2016-12-01 00:00:00
Duplicate months  : 0


,date,rpi,w875rx1,dpcera3m086sbea,cmrmtsplx,retailx,indpro,ipfpnss,ipfinal,ipcongd,ipdcongd,ipncongd,ipbuseq,ipmat,ipdmat,ipnmat,ipmansics,ipb51222s,ipfuels,cumfns,hwi,hwiuratio,clf16ov,ce16ov,unrate,uempmean,uemplt5,uemp5to14,uemp15ov,uemp15t26,uemp27ov,claimsx,payems,usgood,ces1021000001,uscons,manemp,dmanemp,ndmanemp,srvprd,ustpu,uswtrade,ustrade,usfire,usgovt,ces0600000007,awotman,awhman,houst,houstne,houstmw,housts,houstw,permit,permitne,permitmw,permits,permitw,amdmnox,amdmuox,...,ces3000000008,mzmsl,dtcolnvhfnm,dtcthfnm,invest,vxoclsx,a2me,ac,at,ato,beme,beta,c,cf,cf2p,cto,d2a,d2p,dpi2a,e2p,fc2y,idiovol,investment,lev,lme,lt_rev,lturnover,mktbeta,ni,noa,oa,ol,op,pcm,pm,prof,q,r2_1,r12_2,r12_7,r36_13,rel2high,resid_var,rna,roa,roe,s2p,sga2s,spread,st_rev,suv,variance,dp,ep,b/m,ntis,tbl,tms,dfy,svar
0,1967-01-01,-0.000262,-0.001279,0.001592,0.000467,-0.007144,0.002307,0.003122,0.003071,-0.002476,-0.010598,0.000853,0.013831,-0.000003,-0.005039,0.006903,0.004566,-0.002363,-0.005531,-0.104700,-38.000000,-0.069946,0.000405,-0.001504,0.200000,-0.200000,0.053892,-0.040768,0.050431,0.051672,0.048790,0.059308,0.002780,0.000909,0.001471,0.001206,0.000945,0.000905,0.001008,0.003735,0.003050,0.002656,0.004157,0.003671,0.004720,40.600000,-0.300000,40.900000,6.897705,5.187386,5.488938,6.013715,5.075174,6.610696,5.017280,5.170484,5.652489,4.875197,-0.002898,0.003749,...,-0.007561,0.001646,-0.000673,0.011262,0.015138,15.292800,0.000000,0.000000,0.000000,0.000000,0.000000,0.966799,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.248490,0.000000,0.000000,0.000944,0.037798,0.354102,1.088351,0.002740,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.024193,-0.013124,-0.035842,0.037130,0.014519,-0.038626,0.000000,0.000000,0.000000,0.000000,0.000000,0.026759,-0.010710,0.461402,-0.115399,-0.000829,0.003897,0.007481,-0.000341,-0.003600,-0.004100,0.000100,-0.211208
1,1967-02-01,0.009231,0.008679,0.004719,-0.013455,-0.049586,0.004707,0.010373,0.007179,0.009027,-0.025277,0.022914,-0.007057,-0.004207,-0.001682,-0.003099,0.002981,0.025411,0.007101,-0.182100,26.000000,-0.011935,-0.000026,-0.000787,0.100000,-0.200000,0.016301,0.136547,0.002047,0.028371,-0.033902,-0.034462,0.003170,0.002224,-0.000980,0.003310,0.001943,-0.000271,0.005456,0.003651,0.002028,0.001756,0.000045,0.003326,0.005228,40.800000,0.000000,41.100000,6.972606,5.365976,5.659482,6.075346,4.875197,6.902743,5.666427,5.455321,5.774552,5.010635,-0.022886,0.002567,...,0.007561,-0.004710,-0.014617,-0.023986,-0.009671,14.027900,0.000000,0.000000,0.000000,0.000000,0.000000,0.981054,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.107940,0.000000,0.000000,0.132580,0.070439,0.073356,1.168586,0.002740,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-0.010710,-0.030646,-0.056615,0.047345,0.146017,0.078450,0.000000,0.000000,0.000000,0.000000,0.000000,0.105154,0.140489,0.253506,0.087912,-0.071794,-0.081296,-0.078545,-0.001510,-0.002400,-0.002400,-0.000200,0.156913
2,1967-03-01,0.001219,-0.000212,-0.003700,-0.003846,-0.010152,-0.011408,-0.006268,-0.005973,-0.013540,-0.028180,-0.007955,0.001225,-0.019197,-0.028723,-0.010739,-0.011307,0.004764,-0.012178,-1.447600,-31.000000,0.009091,-0.001541,-0.000883,-0.100000,-0.100000,-0.007491,0.009324,-0.063312,-0.103032,-0.009901,0.091388,0.000321,-0.003179,-0.003275,-0.003611,-0.003055,-0.001630,-0.005312,0.002097,0.000390,-0.000223,-0.000347,0.003645,0.003353,40.000000,-0.100000,40.400000,7.023759,5.468060,5.529429,6.104793,5.225747,6.810142,5.298317,5.433722,5.736572,5.123964,-0.005996,0.002314,...,0.003738,0.004705,-0.007544,-0.000001,0.000298,13.988900,0.000000,0.000000,0.000000,0.000000,0.000000,0.978167,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.209591,0.000000,0.000000,-0.136394,-0.037156,-0.011784,1.173965,0.002740,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.140489,0.102

### 10B. Check Missing Values in the Macroeconomic Dataset

After confirming the structure and time coverage of `Macro.csv`, we next examine missing values across the macroeconomic predictors.

This diagnostic is important because macroeconomic time series often begin at different dates or contain occasional missing observations. If missing values are present, they can affect the construction of the macro state variables and later model estimation.

This section reports the fraction of missing observations for each macroeconomic predictor and ranks the variables from highest to lowest missingness.

The purpose is to determine whether:

- the supplied macro dataset is already complete,
- some predictors begin later than others,
- additional preprocessing may be required before the macro variables are used in the LSTM or other deep-learning models.

At this stage, no missing values are automatically filled or removed. The notebook only documents the missing-data structure so that any later treatment is explicit and reproducible.

If the macro dataset contains little or no missing data, it can be used directly in the chronological sample split. If substantial missingness is detected, the appropriate treatment should be decided and documented before model training.

In [24]:

# 10B. Macro missingness report

if macro_raw is not None:
    macro_missing = (
        macro[macro_feature_cols]
        .isna()
        .mean()
        .sort_values(ascending=False)
        .rename("missing_fraction")
        .to_frame()
    )

    print("Macro rows:", f"{len(macro):,}")
    print("Macro columns:", f"{macro.shape[1]:,}")
    display(macro_missing.head(25))
else:
    print("Skipped: macro panel has not been loaded.")


Macro rows: 600
Macro columns: 179


,missing_fraction
rpi,0.000000
cto,0.000000
ddurrg3m086sbea,0.000000
dndgrg3m086sbea,0.000000
dserrg3m086sbea,0.000000
ces0600000008,0.000000
ces2000000008,0.000000
ces3000000008,0.000000
mzmsl,0.000000
dtcolnvhfnm,0.000000


## 11. Construct the Chronological Training, Validation, and Test Samples

The next step is to divide the data into training, validation, and test periods.

Because this is an asset-pricing application, the split must preserve the chronological ordering of the data. A random train-test split would allow observations from later periods to influence model estimation for earlier periods and could introduce look-ahead bias.

Following the replication design, the sample is divided into three non-overlapping periods:

| Sample | Period | Purpose |
|---|---|---|
| Training | 1967–1986 | Estimate model parameters |
| Validation | 1987–1991 | Select model architecture and hyperparameters |
| Test | 1992–2016 | Evaluate true out-of-sample performance |

Formally,

\[$
\text{Training} = \{t: 1967 \leq t \leq 1986\},$
\]

\[$
\text{Validation} = \{t: 1987 \leq t \leq 1991\},$
\]

and

\[$
\text{Test} = \{t: 1992 \leq t \leq 2016\}.$
\]

The validation sample is kept separate from the training sample so that hyperparameter choices can be made without using information from the final test period.

The test sample remains completely out of sample and will be used later to evaluate the economic and statistical performance of the replicated asset-pricing models.

The same date boundaries are applied to both the stock-characteristic dataset and the macroeconomic dataset so that the two information sets remain aligned through time.

### 11A. Define the Sample-Splitting Functions

This subsection defines reusable functions for dividing a dataset according to the fixed chronological boundaries.

The first function assigns each observation to the training, validation, or test sample according to its monthly date.

The second function summarizes each resulting sample by reporting:

- the number of observations,
- the number of unique months,
- the first month,
- the last month.

Because the supplied `RetChar.csv` does not contain a stock identifier, the summary focuses on observations and months rather than the number of unique stocks.

Using the same functions for both stock and macroeconomic data helps ensure that the two datasets are split consistently.

In [25]:
# 11A. Define chronological split functions

def chronological_split(df: pd.DataFrame, date_col: str):
    train = df[df[date_col].between(TRAIN_START, TRAIN_END)].copy()
    valid = df[df[date_col].between(VALID_START, VALID_END)].copy()
    test  = df[df[date_col].between(TEST_START, TEST_END)].copy()

    return train, valid, test


def split_summary(name, train, valid, test, date_col):
    rows = []

    for split_name, part in [
        ("train", train),
        ("validation", valid),
        ("test", test),
    ]:
        rows.append({
            "dataset": name,
            "split": split_name,
            "rows": len(part),
            "months": part[date_col].nunique(),
            "start": part[date_col].min(),
            "end": part[date_col].max(),
        })

    return pd.DataFrame(rows)

### 11B. Apply the Chronological Split to the Stock and Macro Data

After defining the sample-splitting functions, we apply them separately to the processed stock-characteristic dataset and the macroeconomic dataset.

For the stock data, the split is applied using the monthly date variable in `stock_processed`.

For the macro data, the same date boundaries are applied using the monthly date variable in `macro`.

This produces six datasets:

- stock training sample,
- stock validation sample,
- stock test sample,
- macro training sample,
- macro validation sample,
- macro test sample.

A summary table is then created to verify that each dataset covers the intended period and contains the expected number of monthly observations.

The main purpose of this step is to confirm that the stock and macro datasets remain aligned through time and that the final test period is kept completely out of sample.

Any unexpected date gaps or incorrect sample boundaries should be resolved here before the data are used for model estimation.

In [26]:
# 11B. Apply the chronological split

summaries = []

# Split stock-characteristic data
if retchar_raw is not None:
    stock_train, stock_valid, stock_test = chronological_split(
        stock_processed,
        DATE_COL
    )

    summaries.append(
        split_summary(
            "stock_characteristics",
            stock_train,
            stock_valid,
            stock_test,
            DATE_COL
        )
    )


# Split macroeconomic data
if macro_raw is not None:
    macro_train, macro_valid, macro_test = chronological_split(
        macro,
        MACRO_DATE_COL
    )

    summaries.append(
        split_summary(
            "macro",
            macro_train,
            macro_valid,
            macro_test,
            MACRO_DATE_COL
        )
    )


# Display summary
if summaries:
    split_table = pd.concat(
        summaries,
        ignore_index=True
    )

    display(split_table)

else:
    print("No datasets loaded; split step skipped.")

,dataset,split,rows,months,start,end
0,stock_characteristics,train,336113,240,1967-01-01,1986-12-01
1,stock_characteristics,validation,132167,60,1987-01-01,1991-12-01
2,stock_characteristics,test,750275,300,1992-01-01,2016-12-01
3,macro,train,240,240,1967-01-01,1986-12-01
4,macro,validation,60,60,1987-01-01,1991-12-01
5,macro,test,300,300,1992-01-01,2016-12-01


## 12. Leakage and Consistency Checks

Before using the processed data for model estimation, we perform several consistency checks to confirm that the training, validation, and test samples have been constructed correctly.

The first objective is to verify that the three sample periods are completely non-overlapping.

The function `assert_disjoint_dates` converts the dates in the training, validation, and test datasets into separate sets and checks that no month appears in more than one sample.

This ensures that:

- training observations do not appear in the validation sample,
- training observations do not appear in the test sample,
- validation observations do not appear in the test sample.

These checks are particularly important in asset-pricing and machine-learning applications because overlap between samples could introduce look-ahead bias and lead to overly optimistic out-of-sample results.

The same date-overlap check is applied separately to the stock-characteristic dataset and the macroeconomic dataset.

The final part of this section compares the monthly coverage of the two datasets.

We construct the set of months available in the stock-characteristic panel and the set of months available in the macroeconomic panel, and then calculate:

- the number of months shared by both datasets,
- the number of months available only in the stock dataset,
- the number of months available only in the macro dataset.

This comparison is important because the stock-characteristic panel spans 1967–2016, while the supplied macroeconomic dataset begins later, in 1976.

Therefore, models that use both firm characteristics and macroeconomic information can only be estimated over the intersection of the two datasets.

The purpose of this section is to identify these differences explicitly before model training, rather than allowing sample misalignment to occur silently.

In [27]:
# 12. Leakage and consistency checks

def assert_disjoint_dates(train, valid, test, date_col):
    tr = set(train[date_col].dropna().unique())
    va = set(valid[date_col].dropna().unique())
    te = set(test[date_col].dropna().unique())

    assert tr.isdisjoint(va), "Training and validation dates overlap."
    assert tr.isdisjoint(te), "Training and test dates overlap."
    assert va.isdisjoint(te), "Validation and test dates overlap."


# Check stock-characteristic sample
if retchar_raw is not None:
    assert_disjoint_dates(
        stock_train,
        stock_valid,
        stock_test,
        DATE_COL
    )

    print("✓ Stock training, validation, and test dates are disjoint.")
    print("Note: duplicate stock-month checking is not possible because")
    print("RetChar.csv does not contain a stock identifier.")


# Check macro sample
if macro_raw is not None:
    assert_disjoint_dates(
        macro_train,
        macro_valid,
        macro_test,
        MACRO_DATE_COL
    )

    print("✓ Macro training, validation, and test dates are disjoint.")


# Compare stock and macro time coverage
if retchar_raw is not None and macro_raw is not None:

    stock_months = set(
        stock_processed[DATE_COL]
        .dropna()
        .unique()
    )

    macro_months = set(
        macro[MACRO_DATE_COL]
        .dropna()
        .unique()
    )

    shared = stock_months & macro_months
    stock_only = stock_months - macro_months
    macro_only = macro_months - stock_months

    print()
    print(f"Shared stock/macro months : {len(shared):,}")
    print(f"Stock-only months         : {len(stock_only):,}")
    print(f"Macro-only months         : {len(macro_only):,}")

✓ Stock training, validation, and test dates are disjoint.
Note: duplicate stock-month checking is not possible because
RetChar.csv does not contain a stock identifier.
✓ Macro training, validation, and test dates are disjoint.

Shared stock/macro months : 600
Stock-only months         : 0
Macro-only months         : 0



## 13. Save processed outputs

The processed files below are **generated artifacts**, so they belong in `data/processed/`, which is ignored by Git in this repository.

We save Parquet files because they preserve dtypes and are substantially faster and smaller than CSV for large stock panels.


### 13A. Save the Full Processed Datasets

After completing the data-quality checks and preprocessing steps, we save the full processed stock-characteristic and macroeconomic datasets for use in later notebooks.

The stock dataset, `stock_processed`, contains:

- the monthly date,
- the stock return,
- 46 firm characteristics,
- all 1,218,555 stock-month observations.

The macro dataset, `macro`, contains:

- the standardized monthly date,
- 178 macroeconomic predictors,
- observations beginning in January 1976.

The processed datasets are saved in Parquet format rather than CSV.

Parquet is useful for this project because it:

- preserves variable data types,
- loads substantially faster than CSV,
- typically requires less disk space,
- is well suited for large financial datasets.

The processed files are stored in `data/processed/`.

This directory is excluded from GitHub through `.gitignore`, so large generated datasets remain local and are not committed to the repository.

In [28]:
# 13A. Save full processed datasets

if retchar_raw is not None:

    stock_out = PROCESSED_DIR / "retchar_processed.parquet"

    stock_processed.to_parquet(
        stock_out,
        index=False,
        engine="pyarrow",
        compression="snappy"
    )

    print("Saved stock dataset:")
    print(stock_out)


if macro_raw is not None:

    macro_out = PROCESSED_DIR / "macro_processed.parquet"

    macro.to_parquet(
        macro_out,
        index=False,
        engine="pyarrow",
        compression="snappy"
    )

    print("\nSaved macro dataset:")
    print(macro_out)

Saved stock dataset:
/Users/reza/Desktop/mfe-term 3/deepl learning1/project /github/Deep-Learning-in-Asset-Pricing_-Replication-project/data/processed/retchar_processed.parquet

Saved macro dataset:
/Users/reza/Desktop/mfe-term 3/deepl learning1/project /github/Deep-Learning-in-Asset-Pricing_-Replication-project/data/processed/macro_processed.parquet


### 13B. Save the Chronological Training, Validation, and Test Samples

In addition to saving the complete processed datasets, we save the training, validation, and test samples separately.

This avoids repeating the sample-splitting procedure every time a later model notebook is executed.

For the stock-characteristic data, the saved samples correspond to:

- Training: January 1967 – December 1986
- Validation: January 1987 – December 1991
- Test: January 1992 – December 2016

For the macroeconomic data, the same calendar boundaries are used. However, because the supplied macro dataset begins in January 1976, the available macro training sample covers only January 1976 through December 1986.

Saving these datasets separately ensures that later modeling notebooks use exactly the same chronological samples and reduces the possibility of accidental changes to the train-validation-test split.

The resulting Parquet files will serve as the standardized inputs for the benchmark, neural-network, LSTM, and adversarial asset-pricing models developed later in the project.

In [29]:
# 13B. Save train, validation, and test datasets

if retchar_raw is not None:

    stock_splits = {
        "train": stock_train,
        "validation": stock_valid,
        "test": stock_test,
    }

    for split_name, df in stock_splits.items():

        output_path = (
            PROCESSED_DIR /
            f"retchar_{split_name}.parquet"
        )

        df.to_parquet(
            output_path,
            index=False,
            engine="pyarrow",
            compression="snappy"
        )

        print(
            f"Saved stock {split_name}: "
            f"{len(df):,} rows"
        )


if macro_raw is not None:

    macro_splits = {
        "train": macro_train,
        "validation": macro_valid,
        "test": macro_test,
    }

    for split_name, df in macro_splits.items():

        output_path = (
            PROCESSED_DIR /
            f"macro_{split_name}.parquet"
        )

        df.to_parquet(
            output_path,
            index=False,
            engine="pyarrow",
            compression="snappy"
        )

        print(
            f"Saved macro {split_name}: "
            f"{len(df):,} rows"
        )

print("\n✓ All available split datasets saved.")

Saved stock train: 336,113 rows
Saved stock validation: 132,167 rows
Saved stock test: 750,275 rows
Saved macro train: 240 rows
Saved macro validation: 60 rows
Saved macro test: 300 rows

✓ All available split datasets saved.



## 14. Replication checklist

Before moving to the next notebook, verify:

- [ ] `RetChar.csv` loads successfully.
- [ ] `Macro.csv` loads successfully.
- [ ] The stock panel contains the intended 46 firm characteristics.
- [ ] The macro panel is confirmed to represent the intended processed information set.
- [ ] The stock sample covers the intended 1967–2016 period.
- [ ] Complete-case filtering is consistent with the paper replication design.
- [ ] Characteristics are rank-normalized cross-sectionally.
- [ ] Training, validation, and test dates are non-overlapping.
- [ ] Processed Parquet files have been written to `data/processed/`.
- [ ] Any difference from the authors' data construction is documented before modeling.

## Next notebook

**`02_exploratory_analysis.ipynb`**

The next notebook should verify sample sizes, return distributions, characteristic distributions, time coverage, cross-sectional counts, and macroeconomic variation before any model is estimated.
